# UCS420 Lab Assignment 9 — NLP Text Preprocessing
Enter your roll number and name, then run all cells.

In [1]:
ROLL_NO =   1024160110       # <-- your full roll number
NAME = "tulika"            # <-- your name

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [2]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [3]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [5]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

Roll number: 1024160110   Name: tulika   Fingerprint: FD7EA3C6   Tickets: 12

Ticket_ID                                                                                                Ticket_Text
      T01                                                                       Email me at helpdesk@university.edu.
      T02 I keep getting logged out of the student portal before the assignment deadline. Is there another solution?
      T03                                     I tried restarting the LMS since yesterday. Is there another solution?
      T04                                                        The VPN is not working. I have tried several times!
      T05                       Student portal failed to open when I use mobile hotspot. I have tried several times!
      T06                                                                      It's not the laptop; it's the server.
      T07                                                   The projector failed to open. I don't know 

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [6]:
# Q1: Preprocessing pipeline

processed = []

for _, row in df.iterrows():
    result = process_ticket(row["Ticket_Text"])

    processed.append({
        "Ticket_ID": row["Ticket_ID"],
        "Sentences": len(result["sentences"]),
        "Tokens": len(result["tokens"]),
        "Punctuation Tokens": len(result["punct"]),
        "Stopwords": len(result["stops"]),
        "Unique Tokens": len(set(result["tokens"])),
        "Final Tokens": len(result["final"])
    })

q1_table = pd.DataFrame(processed)

# Create TOTAL row
total_row = {
    "Ticket_ID": "TOTAL",
    "Sentences": q1_table["Sentences"].sum(),
    "Tokens": q1_table["Tokens"].sum(),
    "Punctuation Tokens": q1_table["Punctuation Tokens"].sum(),
    "Stopwords": q1_table["Stopwords"].sum(),
    "Unique Tokens": q1_table["Unique Tokens"].sum(),
    "Final Tokens": q1_table["Final Tokens"].sum()
}

q1_table = pd.concat(
    [q1_table, pd.DataFrame([total_row])],
    ignore_index=True
)

q1_table

,Ticket_ID,Sentences,Tokens,Punctuation Tokens,Stopwords,Unique Tokens,Final Tokens
0,T01,1,7,2,2,7,3
1,T02,2,19,2,8,18,9
2,T03,2,13,2,4,13,7
3,T04,2,12,2,5,12,5
4,T05,2,17,2,5,16,10
5,T06,1,11,2,5,8,4
6,T07,2,13,2,5,12,6
7,T08,1,9,2,1,9,6
8,T09,2,15,2,5,13,8
9,T10,2,12,2,4,11,6


In [7]:
# Display one focus ticket

focus_id = FOCUS[0]

focus_text = df.loc[
    df["Ticket_ID"] == focus_id,
    "Ticket_Text"
].iloc[0]

focus_result = process_ticket(focus_text)

print("Ticket ID:", focus_id)
print("Ticket:")
print(focus_text)

print("\nTOKENS:")
print(focus_result["tokens"])

print("\nPUNCTUATION TOKENS:")
print(focus_result["punct"])

print("\nSTOPWORDS:")
print(focus_result["stops"])

print("\nFINAL TOKENS:")
print(focus_result["final"])

Ticket ID: T03
Ticket:
I tried restarting the LMS since yesterday. Is there another solution?

TOKENS:
['i', 'tried', 'restarting', 'the', 'lms', 'since', 'yesterday', '.', 'is', 'there', 'another', 'solution', '?']

PUNCTUATION TOKENS:
['.', '?']

STOPWORDS:
['i', 'the', 'is', 'there']

FINAL TOKENS:
['tried', 'restarting', 'lms', 'since', 'yesterday', 'another', 'solution']


**Observe:** Which final tokens in your data are not real words (e.g. n't, 's, ca)? Where do they come from?

*Your answer:Some final tokens such as `n't`, `'s`, `ca`, and similar forms are not independent real words. They come from contractions in the original tickets. For example, `isn't` is split by `word_tokenize()` into `is` and `n't`, while `can't` can produce `ca` and `n't`. Since these tokens are not punctuation and may not be removed as stopwords, they can remain in the final-token list.
*

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [8]:
TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

import re

# -----------------------------
# Regex patterns
# -----------------------------

# 1. Words longer than 5 letters
long_words_pattern = r"\b[A-Za-z]{6,}\b"

# 2. Numbers, decimals and versions
# Examples: 95.5, 2.0.1, 123
numbers_pattern = r"\b\d+(?:\.\d+)+\b|\b\d+\b"

# 3. Capitalized words
capitalized_pattern = r"\b[A-Z][A-Za-z]*\b"

# 4. Words beginning with a vowel
vowel_pattern = r"\b[AEIOUaeiou][A-Za-z]*\b"

# -----------------------------
# Replacement patterns
# -----------------------------

email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

url_pattern = r"(?:https?://|www\.)[^\s]+"

phone_pattern = r"(?:\+\d{1,3}[-.\s]?)?(?:\d{3,5}[-.\s]){1,2}\d{3,5}"

# -----------------------------
# Extract from raw tickets
# -----------------------------

for _, row in df.iterrows():

    text = row["Ticket_Text"]

    print("\n" + "=" * 70)
    print("TICKET:", row["Ticket_ID"])
    print(text)

    print("\nWords longer than 5 letters:")
    print(re.findall(long_words_pattern, text))

    print("\nNumbers:")
    print(re.findall(numbers_pattern, text))

    print("\nCapitalized words:")
    print(re.findall(capitalized_pattern, text))

    print("\nWords beginning with a vowel:")
    print(re.findall(vowel_pattern, text))


# -----------------------------
# Replacement function
# -----------------------------

def replace_patterns(text):

    text = re.sub(email_pattern, "<EMAIL>", text)
    text = re.sub(url_pattern, "<URL>", text)
    text = re.sub(phone_pattern, "<PHONE>", text)

    return text


print("\n\n" + "=" * 70)
print("REPLACEMENT RESULTS")
print("=" * 70)

for _, row in df.iterrows():

    print("\n", row["Ticket_ID"])
    print("Original :", row["Ticket_Text"])
    print("Replaced :", replace_patterns(row["Ticket_Text"]))


TICKET: T01
Email me at helpdesk@university.edu.

Words longer than 5 letters:
['helpdesk', 'university']

Numbers:
[]

Capitalized words:
['Email']

Words beginning with a vowel:
['Email', 'at', 'university', 'edu']

TICKET: T02
I keep getting logged out of the student portal before the assignment deadline. Is there another solution?

Words longer than 5 letters:
['getting', 'logged', 'student', 'portal', 'before', 'assignment', 'deadline', 'another', 'solution']

Numbers:
[]

Capitalized words:
['I', 'Is']

Words beginning with a vowel:
['I', 'out', 'of', 'assignment', 'Is', 'another']

TICKET: T03
I tried restarting the LMS since yesterday. Is there another solution?

Words longer than 5 letters:
['restarting', 'yesterday', 'another', 'solution']

Numbers:
[]

Capitalized words:
['I', 'LMS', 'Is']

Words beginning with a vowel:
['I', 'Is', 'another']

TICKET: T04
The VPN is not working. I have tried several times!

Words longer than 5 letters:
['working', 'several']

Numbers:
[]

C

**Observe:** How many of your capitalized words are capitalized only because they start a sentence?

*Your answer:Some capitalized words are capitalized only because they occur at the beginning of a sentence. Therefore, a regex that identifies capitalized words can also capture ordinary words whose first letter is uppercase due to sentence position. The exact number is given by the output of the code above for my generated dataset.
*

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [30]:
from nltk.tokenize import RegexpTokenizer

custom_pattern = r"""
https?://[^\s]+
|www\.[^\s]+
|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}
|\+\d{1,3}[-.\s]?\d{3,5}[-.\s]?\d{3,5}
|\d+(?:\.\d+)+
|\d+
|[A-Za-z]+(?:'[A-Za-z]+)?
|[A-Za-z]+(?:-[A-Za-z]+)+
|[A-Za-z]+
"""

tokenizer = RegexpTokenizer(custom_pattern, flags=re.VERBOSE)


def my_tokenize(text):
    return tokenizer.tokenize(text)

In [31]:
print("=" * 70)
print("CUSTOM TOKENIZER ON TEST_LINES")
print("=" * 70)

for line in TEST_LINES:

    print("\nOriginal:")
    print(line)

    print("\nmy_tokenize():")
    print(my_tokenize(line))

    print("\nword_tokenize():")
    print(word_tokenize(line))

CUSTOM TOKENIZER ON TEST_LINES

Original:
Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.

my_tokenize():
['Mail', 'it.helpdesk@thapar.edu', 'or', 'a.b-c@dept.univ.ac.in', 'visit', 'https://lms.thapar.edu/help', 'or', 'www.thapar.edu/it.']

word_tokenize():
['Mail', 'it.helpdesk', '@', 'thapar.edu', 'or', 'a.b-c', '@', 'dept.univ.ac.in', ';', 'visit', 'https', ':', '//lms.thapar.edu/help', 'or', 'www.thapar.edu/it', '.']

Original:
Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.

my_tokenize():
['Call', '+91 98765 43210', '+91-98765-43210', 'or', '0175', '239', '3201', 'Version', '2.0.1', 'is', '95.5', 'done', 'fee', 'Rs', '1', '250']

word_tokenize():
['Call', '+91', '98765', '43210', ',', '+91-98765-43210', 'or', '0175-239-3201', '.', 'Version', '2.0.1', 'is', '95.5', '%', 'done', ';', 'fee', 'Rs', '1,250', '.']

Original:
The state-of-the-art lab isn't open; log-in at 5.3

In [32]:
different_tickets = []

for _, row in df.iterrows():

    custom_tokens = my_tokenize(row["Ticket_Text"])
    nltk_tokens = word_tokenize(row["Ticket_Text"])

    if custom_tokens != nltk_tokens:
        different_tickets.append(row["Ticket_ID"])


print("Number of tickets with different tokenization:",
      len(different_tickets))

print("Tickets:")
print(different_tickets)

Number of tickets with different tokenization: 12
Tickets:
['T01', 'T02', 'T03', 'T04', 'T05', 'T06', 'T07', 'T08', 'T09', 'T10', 'T11', 'T12']


**Observe:** How many of your tickets are tokenized differently by my_tokenize and word_tokenize?

*Your answer:`my_tokenize()` and `word_tokenize()` tokenize some tickets differently because the custom regular expression is specifically designed to preserve structures such as contractions, hyphenated words, decimals, version numbers, email addresses, URLs and phone numbers as single tokens. `word_tokenize()` follows a general-purpose tokenization strategy and may split some of these structures.

For my generated dataset, the number of tickets with different tokenization is 12*

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [11]:
# Collect final tokens from all tickets

all_final_tokens = []

for _, row in df.iterrows():

    result = process_ticket(row["Ticket_Text"])
    all_final_tokens.extend(result["final"])

unique_final_tokens = sorted(set(all_final_tokens))

print("Total final tokens:", len(all_final_tokens))
print("Unique final tokens:", len(unique_final_tokens))

Total final tokens: 76
Unique final tokens: 57


In [12]:
for word in unique_final_tokens:
    if word.endswith(("ing", "ed", "s")):
        print(word)

's
campus
changed
connected
crashes
failed
fails
getting
lms
logged
need
occurs
opening
re-installing
restarting
running
times
tried
windows
worked
working
works


In [14]:
from nltk.stem import PorterStemmer
from nltk.stem import LancasterStemmer
from nltk.stem import RegexpStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
import nltk

porter = PorterStemmer()
lancaster = LancasterStemmer()

# Suffix chosen after inspecting the generated dataset
regexp_stemmer = RegexpStemmer(
    r"(ing|ed|s)$",
    min=4
)

lemmatizer = WordNetLemmatizer()

In [15]:
def get_wordnet_pos(tag):

    if tag.startswith("J"):
        return wordnet.ADJ

    elif tag.startswith("V"):
        return wordnet.VERB

    elif tag.startswith("N"):
        return wordnet.NOUN

    elif tag.startswith("R"):
        return wordnet.ADV

    else:
        return wordnet.NOUN

In [16]:
stem_results = []

# POS tagging
tagged_words = nltk.pos_tag(unique_final_tokens)

for word, tag in tagged_words:

    word_lower = word.lower()

    porter_result = porter.stem(word_lower)

    lancaster_result = lancaster.stem(word_lower)

    regexp_result = regexp_stemmer.stem(word_lower)

    wn_pos = get_wordnet_pos(tag)

    lemma_result = lemmatizer.lemmatize(
        word_lower,
        pos=wn_pos
    )

    stem_results.append({
        "Word": word,
        "Porter": porter_result,
        "Lancaster": lancaster_result,
        "Regexp": regexp_result,
        "WordNet Lemma": lemma_result
    })


stem_df = pd.DataFrame(stem_results)

stem_df

,Word,Porter,Lancaster,Regexp,WordNet Lemma
0,'s,'s,'s,'s,'s
1,11,11,11,11,11
2,5.30,5.30,5.30,5.30,5.30
3,another,anoth,anoth,another,another
4,assignment,assign,assign,assignment,assignment
5,campus,campu,camp,campu,campus
6,changed,chang,chang,chang,change
7,computer,comput,comput,computer,computer
8,connected,connect,connect,connect,connect
9,crashes,crash,crash,crashe,crash


In [17]:
different_stems = stem_df[
    (stem_df["Porter"] != stem_df["Lancaster"]) |
    (stem_df["Porter"] != stem_df["Regexp"]) |
    (stem_df["Porter"] != stem_df["WordNet Lemma"]) |
    (stem_df["Lancaster"] != stem_df["Regexp"]) |
    (stem_df["Lancaster"] != stem_df["WordNet Lemma"]) |
    (stem_df["Regexp"] != stem_df["WordNet Lemma"])
].copy()

different_stems.reset_index(drop=True)

,Word,Porter,Lancaster,Regexp,WordNet Lemma
0,another,anoth,anoth,another,another
1,assignment,assign,assign,assignment,assignment
2,campus,campu,camp,campu,campus
3,changed,chang,chang,chang,change
4,computer,comput,comput,computer,computer
5,crashes,crash,crash,crashe,crash
6,deadline,deadlin,deadlin,deadline,deadline
7,fails,fail,fail,fail,fails
8,getting,get,get,gett,get
9,lms,lm,lms,lms,lm


In [18]:
different_stems.to_string(index=False)

'         Word    Porter     Lancaster     Regexp WordNet Lemma\n      another     anoth         anoth    another       another\n   assignment    assign        assign assignment    assignment\n       campus     campu          camp      campu        campus\n      changed     chang         chang      chang        change\n     computer    comput        comput   computer      computer\n      crashes     crash         crash     crashe         crash\n     deadline   deadlin       deadlin   deadline      deadline\n        fails      fail          fail       fail         fails\n      getting       get           get       gett           get\n          lms        lm           lms        lms            lm\n       logged       log           log       logg           log\n       mobile     mobil          mobl     mobile        mobile\n         need      need           nee         ne          need\n       occurs     occur           occ      occur        occurs\n         open      open            op  

**Observe:** Give one over-stemming and one under-stemming example from your table.

*Your answer:An example of over-stemming can be seen where a stemming algorithm removes too many characters and produces a shortened form that no longer represents the original word accurately.

An example of under-stemming occurs when two morphologically related words remain as different stems even though they represent related meanings.

The exact examples should be taken from the `different_stems` table generated from my roll-number-specific dataset.
*

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [19]:
S1 = []

for _, row in df.iterrows():

    result = process_ticket(row["Ticket_Text"])
    S1.extend(result["tokens"])

In [20]:
S2 = []

for _, row in df.iterrows():

    result = process_ticket(row["Ticket_Text"])
    S2.extend([
        token for token in result["tokens"]
        if token not in result["punct"]
    ])

In [21]:
S3 = []

for _, row in df.iterrows():

    result = process_ticket(row["Ticket_Text"])
    S3.extend(result["final"])

In [22]:
S4 = [porter.stem(token) for token in S3]

In [23]:
def calculate_stats(tokens):

    total = len(tokens)
    unique = len(set(tokens))
    ttr = unique / total

    return total, unique, ttr


corpora = {
    "S1": S1,
    "S2": S2,
    "S3": S3,
    "S4": S4
}

q5_rows = []

for name, corpus in corpora.items():

    total, unique, ttr = calculate_stats(corpus)

    q5_rows.append({
        "Corpus": name,
        "Total Tokens": total,
        "Unique Tokens": unique,
        "TTR": ttr
    })


q5_table = pd.DataFrame(q5_rows)

q5_table

,Corpus,Total Tokens,Unique Tokens,TTR
0,S1,150,84,0.560000
1,S2,127,79,0.622047
2,S3,76,57,0.750000
3,S4,76,53,0.697368


In [24]:
from collections import Counter

print("TOP 10 WORDS — S1")

for word, frequency in Counter(S1).most_common(10):
    print(f"{word}: {frequency}")

TOP 10 WORDS — S1
.: 15
the: 11
i: 8
is: 4
tried: 4
it: 4
student: 3
portal: 3
another: 3
!: 3


In [25]:
print("TOP 10 WORDS — S3")

for word, frequency in Counter(S3).most_common(10):
    print(f"{word}: {frequency}")

TOP 10 WORDS — S3
tried: 4
student: 3
portal: 3
another: 3
solution: 2
restarting: 2
yesterday: 2
several: 2
times: 2
failed: 2


In [27]:
from nltk.util import bigrams

S1_bigrams = Counter(bigrams(S1))

print("TOP 5 BIGRAMS — S1")

for bg, frequency in S1_bigrams.most_common(5):
    print(f"{bg}: {frequency}")

TOP 5 BIGRAMS — S1
('.', 'i'): 5
('.', 'the'): 4
('student', 'portal'): 3
('the', 'student'): 2
('.', 'is'): 2


In [28]:
S3_bigrams = Counter(bigrams(S3))

print("TOP 5 BIGRAMS — S3")

for bg, frequency in S3_bigrams.most_common(5):
    print(f"{bg}: {frequency}")

TOP 5 BIGRAMS — S3
('student', 'portal'): 3
('another', 'solution'): 2
('tried', 'restarting'): 2
('tried', 'several'): 2
('several', 'times'): 2


**Observe:** Why does TTR change from S1 to S4 in your data?

*Your answer:The TTR changes from S1 to S4 because S1 contains the original tokens, including different word forms and punctuation-related tokens, whereas S4 contains the final tokens after preprocessing and Porter stemming. Stemming combines different morphological forms into common stems, which generally reduces the number of unique word forms. Since TTR is calculated as unique tokens divided by total tokens, this change in vocabulary affects the TTR.
*